In [1]:
import os
import requests
import zipfile

url = "https://www.dropbox.com/scl/fi/wruzj2bwyg743d0jzd7ku/all-the-news-3.zip?rlkey=rgwtwpeznbdadpv3f01sznwxa&dl=1"
zip_path = "./data/all-the-news-3.zip"
csv_path = "./data/all-the-news-3.csv"

os.makedirs("./data", exist_ok=True)

# Check if CSV file already exists
if os.path.exists(csv_path):
    print(f"CSV file already exists: {csv_path}")
else:
    # Check if ZIP file already exists
    if not os.path.exists(zip_path):
        print("Downloading dataset...")
        response = requests.get(url, stream=True)
        response.raise_for_status()
        with open(zip_path, "wb") as f:
            for chunk in response.iter_content(chunk_size=8192):
                if chunk:
                    f.write(chunk)
        print("Download completed:", zip_path)
    else:
        print(f"ZIP file already exists: {zip_path}")

    # Extract ZIP only if CSV doesn’t exist
    print("Extracting ZIP file...")
    with zipfile.ZipFile(zip_path, "r") as zip_ref:
        zip_ref.extractall("./data")
    print("Extraction completed to ./data")

CSV file already exists: ./data/all-the-news-3.csv


In [2]:
import pandas as pd
from api_utils import Utils
from huggingface_hub import InferenceClient
from sentence_transformers import SentenceTransformer
from langchain.text_splitter import RecursiveCharacterTextSplitter
from tqdm.auto import tqdm

# Dataset

In [3]:
df = pd.read_csv(csv_path, engine='python', on_bad_lines='skip')
df

,date,year,month,day,author,title,article,url,section,publication
0,2016-12-09 18:31:00,2016,12.0,9,Lee Drutman,We should take concerns about the health of li...,"This post is part of Polyarchy, an independent...",https://www.vox.com/polyarchy/2016/12/9/138983...,NaN,Vox
1,2016-10-07 21:26:46,2016,10.0,7,Scott Davis,Colts GM Ryan Grigson says Andrew Luck's contr...,The Indianapolis Colts made Andrew Luck the h...,https://www.businessinsider.com/colts-gm-ryan-...,NaN,Business Insider
2,2018-01-26 00:00:00,2018,1.0,26,NaN,Trump denies report he ordered Mueller fired,"DAVOS, Switzerland (Reuters) - U.S. President ...",https://www.reuters.com/article/us-davos-meeti...,Davos,Reuters
3,2019-06-27 00:00:00,2019,6.0,27,NaN,France's Sarkozy reveals his 'Passions' but in...,PARIS (Reuters) - Former French president Nico...,https://www.reuters.com/article/france-politic...,World News,Reuters
4,2016-01-27 00:00:00,2016,1.0,27,NaN,Paris Hilton: Woman In Black For Uncle Monty's...,Paris Hilton arrived at LAX Wednesday dressed ...,https://www.tmz.com/2016/01/27/paris-hilton-mo...,NaN,TMZ
...,...,...,...,...,...,...,...,...,...,...
106477,2017-03-23 00:00:00,2017,3.0,23,NaN,Kristi Yamaguchi to Nancy Kerrigan -- No Bad B...,Kristi Yamaguchi says she was NOT throwing sha...,https://www.tmz.com/2017/03/23/kristi-yamaguch...,NaN,TMZ
106478,2017-05-30 00:00:00,2017,5.0,30,Claire Voon,"Facing Protests from Islamist Groups, Banglade...",A statue of Lady Justice in front of the Supre...,https://hyperallergic.com/382201/facing-protes...,NaN,Hyperallergic
106479,2018-04-24 15:20:02,2018,4.0,24,Constance Grady,Laura and Emma review: Kate Greathead’s novel ...,"Laura and Emma, a debut novel from Kate Greath...",https://www.vox.com/culture/2018/4/24/17225200...,NaN,Vox
106480,2017-10-17 17:13:00,2017,10.0,17,Meredith Hoffman,Trans Advocates Won't Thank Sessions for a Hat...,"Last week, Attorney General Jeff Sessions ass...",https://www.vice.com/en_us/article/evpj9j/tran...,Identity,Vice


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 106482 entries, 0 to 106481
Data columns (total 10 columns):
 #   Column       Non-Null Count   Dtype 
---  ------       --------------   ----- 
 0   date         106482 non-null  object
 1   year         106474 non-null  object
 2   month        106474 non-null  object
 3   day          106474 non-null  object
 4   author       76727 non-null   object
 5   title        106474 non-null  object
 6   article      106151 non-null  object
 7   url          106473 non-null  object
 8   section      47752 non-null   object
 9   publication  106473 non-null  object
dtypes: object(10)
memory usage: 8.1+ MB


In [5]:
df_nnull = df.dropna()
df_nnull = df_nnull.rename(columns={'article': 'content'})
df_nnull.info()

<class 'pandas.core.frame.DataFrame'>
Index: 33338 entries, 7 to 106481
Data columns (total 10 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   date         33338 non-null  object
 1   year         33338 non-null  object
 2   month        33338 non-null  object
 3   day          33338 non-null  object
 4   author       33338 non-null  object
 5   title        33338 non-null  object
 6   content      33338 non-null  object
 7   url          33338 non-null  object
 8   section      33338 non-null  object
 9   publication  33338 non-null  object
dtypes: object(10)
memory usage: 2.8+ MB


In [6]:
SAMPLING_SIZE = 5000
df_sampled = df_nnull.sample(n=SAMPLING_SIZE, random_state=42).reset_index(drop=True)
len(df_sampled)

5000

# Store Embeddings

In [7]:
embedding_model_name = 'all-MiniLM-L6-v2'
embedding_model = SentenceTransformer(embedding_model_name)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [8]:
EMBEDDING_DIM = embedding_model.get_sentence_embedding_dimension()
MAX_SEQ_LENGTH = embedding_model.max_seq_length

EMBEDDING_DIM, MAX_SEQ_LENGTH

(384, 256)

- **In the `content` column, we will split each row into chunks of texts, and we will tokenize each chunk**
- **Each chunk after tokenization must ensure its length <= `MAX_SEQ_LENGTH` of model (so that positional encoding will work)**

In [9]:
title_index = Utils.create_index('title-index', dimension=EMBEDDING_DIM)
content_index = Utils.create_index('content-index', dimension=EMBEDDING_DIM)

Index 'title-index' already exists → deleting ...
Deleted 'title-index' successfully!
Creating index 'title-index' ...
Index 'title-index' created successfully!
Index 'content-index' already exists → deleting ...
Deleted 'content-index' successfully!
Creating index 'content-index' ...
Index 'content-index' created successfully!


In [10]:
def upsert_title(df, index, embedding_model, batch_size=512):

    for i in tqdm(range(0, len(df), batch_size)):
        i_end = min(i + batch_size, len(df))

        ids = [str(j) for j in range(i, i_end)]
        titles = df['title'].iloc[i:i_end].tolist()
        values = embedding_model.encode(titles)
        meta_titles = [
            {'title_id': f"{j}/{len(df)}",'title': t}
            for j, t in zip(range(i, i_end), titles)
        ]

        records = zip(ids, values, meta_titles)
        records = list(
            map(lambda r: {'id': r[0], 'values': r[1], 'metadata': r[2]}, records)
        )

        index.upsert(vectors=records)

In [11]:
def upsert_content(df, index, embedding_model, batch_size=512, chunk_size_ratio=0.8, chunk_overlap_ratio=0.1):
    MAX_SEQ_LENGTH = embedding_model.max_seq_length
    CHUNK_SIZE = int(MAX_SEQ_LENGTH * chunk_size_ratio)
    CHUNK_OVERLAP = int(CHUNK_SIZE * chunk_overlap_ratio)
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)

    batch_to_upsert = []

    for i, row in tqdm(df.iterrows(), total=len(df), desc='Upserting content of each article in chunks'):
        title = row['title']
        chunks = text_splitter.split_text(row['content'])

        ids = []
        values = embedding_model.encode(chunks)
        meta_title_id = [f"{i}/{len(df)}"] * len(chunks)
        meta_title = [title] * len(chunks)
        meta_chunk_ids = []
        meta_chunk_texts = chunks

        for j in range(len(chunks)):
            ids.append(f"{i}-{j}")
            meta_chunk_ids.append(f"{j}/{len(chunks)}")


        records = zip(ids, values, meta_title_id, meta_title, meta_chunk_ids, meta_chunk_texts)
        records = list(
            map(lambda r: {'id': r[0], 'values': r[1], 'metadata': {'title_id': r[2], 'title': r[3], 'chunk_id': r[4], 'chunk_text': r[5]}},
                records)
        )

        batch_to_upsert.extend(records)

        if len(batch_to_upsert) >= batch_size:
            index.upsert(vectors=batch_to_upsert)
            batch_to_upsert = []

    # upsert last batch, its size may < batch_size
    if len(batch_to_upsert) > 0:
        index.upsert(vectors=batch_to_upsert)

In [12]:
upsert_title(df_sampled, title_index, embedding_model, batch_size=768)

  0%|          | 0/7 [00:00<?, ?it/s]

In [13]:
upsert_content(df_sampled, content_index, embedding_model, batch_size=768)

Upserting content of each article in chunks:   0%|          | 0/5000 [00:00<?, ?it/s]

In [14]:
title_index.describe_index_stats(), content_index.describe_index_stats()

({'dimension': 384,
  'index_fullness': 0.0,
  'metric': 'cosine',
  'namespaces': {'': {'vector_count': 5000}},
  'total_vector_count': 5000,
  'vector_type': 'dense'},
 {'dimension': 384,
  'index_fullness': 0.0,
  'metric': 'cosine',
  'namespaces': {'': {'vector_count': 115222}},
  'total_vector_count': 115222,
  'vector_type': 'dense'})

# Recommendations

In [15]:
from collections import defaultdict

In [16]:
def get_recommendations_from_title(query, index=title_index, embedding_model=embedding_model, top_k=5):
    values = embedding_model.encode(query).tolist()
    results = index.query(vector=values, top_k=top_k, include_metadata=True, include_values=False)

    recommendations = [
        (m.metadata.get('title', 'Unknown'), m.score)
        for m in results.matches
    ]
    return recommendations

In [17]:
def get_recommendations_from_content(query, index=content_index, embedding_model=embedding_model, top_k=20, alpha=0.3):
    """
    final_score = max_score + alpha * mean_score
    """

    values = embedding_model.encode(query).tolist()
    results = index.query(vector=values, top_k=top_k, include_metadata=True, include_values=False)

    # {'title_A': [s1, s2, s3], 'title_B': [s1, s2], ...}
    title_scores = defaultdict(list)
    for m in results.matches:
        title_scores[m.metadata['title']].append(m.score)

    agg_title_scores = {}
    for title, score_list in title_scores.items():
        max_score = max(score_list)
        mean_score = sum(score_list) / len(score_list)
        agg_title_scores[title] = (1-alpha) * max_score + alpha * mean_score

    recommendations = sorted(agg_title_scores.items(), key=lambda x: x[1], reverse=True)
    return recommendations

In [18]:
def print_recommendations(query):
    title_recs = get_recommendations_from_title(query)
    print(f'Title Retrieval for "{query}"')
    print("-" * 40)
    for i, (title, score) in enumerate(title_recs, start=1):
        print(f"{i:<1}. {title:<90}  —  score: {score:.4f}")

    print("\n")

    content_recs = get_recommendations_from_content(query)
    print(f'Content Retrieval for "{query}"')
    print("-" * 40)
    for i, (title, score) in enumerate(content_recs[:len(title_recs)], start=1):
        print(f"{i:<1}. {title:<90}  —  score: {score:.4f}")

In [19]:
queries = [
    "Advances in Generative AI",                           # Technology & AI
    "Quantum computing breakthroughs in 2025",             # Computer science & physics
    "Renewable energy and climate change solutions",       # Environment & energy
    "AI applications in healthcare and disease detection", # Medicine & health
    "The future of automation and jobs",                   # Economy & labor
    "The role of AI in art and creativity",                # Art & creativity
    "Social media algorithms and misinformation",          # Society & communication
    "Breakthroughs in gene therapy",                       # Biology & biotechnology
    "Virtual reality and immersive experiences",           # Entertainment & technology
    "Cybersecurity and data privacy in modern society"     # Security & law
]


for q in queries:
    print_recommendations(q)
    print("\n")

Title Retrieval for "Advances in Generative AI"
----------------------------------------
1. AI Could Usher in a New Generation of Catfishing                                            —  score: 0.4557
2. AI Has Made Video Surveillance Automated and Terrifying                                     —  score: 0.3626
3. Neuroscientists Have a New Computational Model for Memory                                   —  score: 0.3455
4. Faces for cookware: data collection industry flourishes as China pursues AI ambitions       —  score: 0.3436
5. Scientists Invented AI Made From DNA                                                        —  score: 0.3265


Content Retrieval for "Advances in Generative AI"
----------------------------------------
1. These People Are Not Real—They Were Created By AI                                           —  score: 0.5516
2. AI Could Usher in a New Generation of Catfishing                                            —  score: 0.4678
3. Watch a Computer Learn to Play 